# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [1]:
%load_ext autoreload
%autoreload 2

import json
from _campaign_lib import *

# --- Services ---
svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
campaign_config = {
    "sample_size": 15,              # queries per eval step (0 = all)
    "exploration_sample_size": 10,  # queries per scan/grid point (can be smaller)
    "exploration_rate": 0.5,        # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": None,              # default: 10 (None = unlimited)
        "degradation_threshold": 0.4,    # fraction of degraded queries to trigger escalation
        "backend_warning_threshold": 2,  # degradation resets before backend advisory
        "enable_l2": True,               # L2 refine_context on escalation
        "enable_l3": True,               # L3 modify_plan on L2 stall
        "l2_patience": None,             # default: 2
        "l3_patience": None,             # default: 1
    },
    "eval_llm": {
        "model":       "moonshotai/kimi-k2-instruct-0905",
        "provider":    "groq",
        "temperature": 0.4,
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",
        # "model": "claude-sonnet-4-6",
        # "model": "claude-haiku-4-5-20251001",
        "max_tokens": 2000,
    },
    "pipeline_params": None        # set by configure_pipeline()
}

# --- Pipeline snapshot & params ---
pipeline_config_full = await show_pipeline_snapshot(svc)
pipeline_params = configure_pipeline(svc, campaign_config)

Backend: http://127.0.0.1:8000


2026-03-25 10:30:29 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-25 10:30:29 INFO     [api.services.pipeline_discovery] Parsed pipeline 'TermNorm' with 6 steps
2026-03-25 10:30:29 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1
2026-03-25 10:30:29 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


Pipeline: termnorm (6 steps)
Experiment: production_historical (40 queries, 93 session terms)
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md
  PIPELINE SNAPSHOT: TermNorm v1.1
  Nodes:   ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']

{
  "name": "TermNorm",
  "version": "v1.1",
  "description": "TermNorm AI terminology normalization pipeline",
  "required_step": "entity_profile",
  "template_variables": [
    "{{core_concept}}",
    "{{entity_profile_json}}",
    "{{matches}}"
  ],
  "dataset_name": "termnorm_ground_truth",
  "available_models": [
    "moonshotai/kimi-k2-instruct-0905",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct"

In [ ]:
#@title Load data & evaluation context
RUN_BASELINE = False

train_data, session_terms = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx",
)
svc["session_terms"] = session_terms

baseline_ps, eval_data, campaign_rounds, baseline_results = await prepare_eval_context(
    svc, train_data, campaign_config, run_baseline=RUN_BASELINE,
)

In [ ]:
#@title Experiment dashboard
EXPERIMENT_ID = None  # Set to hex ID to resume (e.g. '68e2c5')

pipeline_params = show_experiment_dashboard(
    svc=svc, experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config, eval_data=eval_data,
    baseline_prompt_state=campaign_rounds[0]["prompt_state"].model_dump() if campaign_rounds else None,
)

## 3. Explore

Exploration via **Smart Search** (scan advisor + sensitivity scan).

In [ ]:
#@title Task context + scan advisor
task_context = await decompose_task_context(TASK_DESCRIPTION, campaign_config, svc)

# preview_advisor_prompt(campaign_config, svc, task_description=task_context, raw=True)
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=task_context,
)

In [8]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 3  # queries per scan variant (0 = use all)

scan_variants = {
    # ── Token matching ───────────────────────────────────────────────────
    'max_token_candidates': [10, 30, 50],
    # ── Web search: query framing ────────────────────────────────────────
    'query_prefix': [
        # --- Database-oriented ---
        'ecoinvent', 'GaBi', 'ecoinvent LCA database material name',
        'material composition LCA', # 'ecoinvent equivalent', 'ecoinvent match for', 'identify LCA material for',
        'what is',# 'define', 'identify', 'describe material',
        # 'chemical composition of', 'CAS number', 'IUPAC name for',
        'technical data sheet', 'product specification', # 'manufacturer datasheet', 'material safety data sheet',
        'manufactured from', 'production process for', # 'raw material for',
        # --- Synonym / translation ---
        # 'also known as', # 'synonym for', 'equivalent material', 'alternative name for',
        # 'wikipedia', # 'material properties of',
    ],
    # ── Web search: volume knobs ─────────────────────────────────────────
    'max_sites': [3, 7, 12],
    'num_results': [5, 20, 40],
    'content_char_limit': [400, 800, 1500],
    # ── Fuzzy matching ───────────────────────────────────────────────────
    # 'fuzzy_threshold': [50, 70, 90],
    # 'fuzzy_scorer': ['ratio', 'WRatio', 'token_set_ratio'],
    # ── Entity profiling: LLM tuning ─────────────────────────────────────
    'profiling_temperature': [0.0, 0.3, 0.7],
    'raw_content_limit': [1000, 2500, 8000],
    # ── Prompt fields ────────────────────────────────────────────────────
    'thinking_style': [
          'Think step-by-step: isolate distinguishing features → compare each candidate → assign scores.',
          'Consider the most likely interpretation first, then check alternatives.',       
          'Reason by elimination: discard obviously wrong candidates, then rank the rest.',
      ],
    # ── Entity profiling: schema mutations ───────────────────────────────
    'profiling_schema': [
        # --- LCA-database-specific (original) ---
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'],
         ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]],
        [['-', 'manufacturing_processes'], ['-', 'applications'],
         ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        # [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]],
        [['-', 'applications'], ['+', 'ecoinvent_candidate_names', 'array', True, 'Exact ecoinvent activity names this entity most likely maps to']],
        [['-', 'manufacturing_processes'], ['+', 'database_search_tokens', 'array', True, 'Optimized search tokens for LCA database lookup including spelling variants']],
        # [['~', 'classification_aliases', 'classification_aliases', 'array', True, 'All valid ecoinvent/GaBi naming variants including geography codes and system model suffixes']],
        # --- Minimalist: strip to core matching signals ---
        [['-', 'applications'], ['-', 'manufacturing_processes'], ['-', 'notes'], ['-', 'technical_specifications']],
        # --- Chemical identity ---
        [['+', 'cas_number', 'string', False, 'CAS registry number if identifiable from context'],
         ['+', 'chemical_formula', 'string', False, 'Chemical formula or molecular structure notation']],
        # --- Trade name decoding ---
        [['~', 'key_properties', 'trade_names', 'array', True, 'Known commercial/trade names and brand names for this material, e.g. Makrolon=polycarbonate, Delrin=POM']],
        # --- Material hierarchy (specific→generic) ---
        [['+', 'material_hierarchy', 'array', False, 'Classification chain from specific to generic, e.g. [Makrolon 2805, polycarbonate, thermoplastic, polymer]']],
        # --- Process-centric (flip perspective from material to process) ---
        [['~', 'applications', 'production_route', 'string', False, 'Primary production/manufacturing route e.g. injection molding, extrusion, casting'],
         ['~', 'notes', 'form_factor', 'string', False, 'Physical form: granulate, sheet, rod, wire, powder, liquid, film']],
        # --- Standards-focused ---
        [['+', 'applicable_standards', 'array', False, 'DIN/ISO/EN/ASTM standards that reference or define this material'],
         ['-', 'applications']],
        # # --- Geography-aware ---
        # [['+', 'supply_chain_geography', 'string', False, 'Most likely geographic origin or market region for this material']],
        # # --- Confidence / ambiguity signal ---
        # [['+', 'confidence_level', 'string', False, 'How confident the model is in the identification: high/medium/low/ambiguous'],
        #  ['+', 'ambiguity_notes', 'string', False, 'What makes this input hard to identify — abbreviation, trade name, multi-material, etc.']],
        # # --- Spelling / language variant boost ---
        # [['+', 'spelling_variants', 'array', False, 'All known spelling variants across EN/DE/FR, e.g. aluminium/aluminum, polyamid/polyamide'],
        #  ['-', 'notes']],
        # --- Werkstoff / alloy code decoding ---
        [['+', 'material_code_decoded', 'string', False, 'Decoded meaning of any material code, Werkstoff number, or alloy designation present in the input'],
         ['+', 'base_material', 'string', False, 'The fundamental base material, e.g. brass, steel, polycarbonate']],
        # --- Functional equivalence ---
        [['~', 'applications', 'functional_unit', 'string', False, 'The functional unit this material serves, e.g. structural plastic, electrical insulation, food-grade packaging'],
         ['+', 'substitutes', 'array', False, 'Materials that could serve the same functional role']],
    ],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['ecoinvent', 'GaBi', 'ecoinvent LCA database material name', 'material composition LCA', 'what is', 'technical data sheet', 'product specification', 'manufactured from', 'production process for']
  max_sites: [3, 7, 12]
  num_results: [5, 20, 40]
  content_char_limit: [400, 800, 1500]
  profiling_temperature: [0.0, 0.3, 0.7]
  raw_content_limit: [1000, 2500, 8000]
  thinking_style: ['Think step-by-step: isolate distinguishing features → compare each candidate → assign scores.', 'Consider the most likely interpretation first, then check alternatives.', 'Reason by elimination: discard obviously wrong candidates, then rank the rest.']
  profiling_schema: (baseline + 13 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this e

In [ ]:
#@title Run sensitivity scan
scan_baseline_sp, scan_df, axis_profiles = await run_sensitivity_scan(
    baseline_ps, campaign_config, scan_variants, eval_data,
    scan_sample_size=scan_sample_size,
    svc=svc, experiment_id=EXPERIMENT_ID or "",
)

In [ ]:
#@title Scan analytics
difficulty_df = show_scan_analytics(scan_df, axis_profiles, svc)

In [ ]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, scan_baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

2026-03-24 13:45:17 INFO     [api.services.search.smart_search] select_scan_winner: 0 prompt changes, 2 param changes from 2 improving axes


Selected best from 2 improving axes:
  query_prefix              best_delta=+30.0%  value_idx=3  acc=66.7%
  profiling_temperature     best_delta=+30.0%  value_idx=0  acc=66.7%
Pipeline params updated: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'query_prefix': 'material composition LCA', 'profiling_temperature': 0.0}
Updated pipeline_params: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'query_prefix': 'material composition LCA', 'profiling_temperature': 0.0}

Round    Accuracy   Rolling Avg    Trend
  search    33.3%        33.3%  -


## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=scan_df,
    axis_profiles=axis_profiles,
    scan_variants=scan_variants,
    difficulty_df=locals().get("difficulty_df"),
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 33.3%
  Baseline prompt        : 1) Extract the entity_category and key distinguishing features from the profile....
  ------------------------------------------------------------------
  Max rounds             : unlimited
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : enabled, patience=2
  L3 (modify plan)       : enabled, patience=1
  ------------------------------------------------------------------
  Candidate model        : moonshotai/kimi-k2-instruct-0905
  Creativity             : 0.7
  Pipeline               : 5 of 6 steps
    Steps                : cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
 

In [ ]:
#@title Run optimization (feedback cycle)
dev_reload()

campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    svc=svc, pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=EXPERIMENT_ID,
    task_context=task_context,
)

In [ ]:
#@title 5. Results — summary, save, sync
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

# --- Persist (T2: below the fold) ---
save_campaign_winner(
    campaign_rounds, campaign_config, svc["store"], svc["backend_id"],
    experiment_id=EXPERIMENT_ID,
)
sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)